# XOPT - OR-LIBRARY p-MEDIAN WITH MIP

It solves every OR-Library p-median instance with the exact `python-mip` formulation, reporting runtime, solver status, and gap to the published best-known value.

### SETUP

Importing the libraries:

In [1]:
import gc
import os
import sys

import numpy  as np
import pandas as pd

from IPython.display import display
from time            import perf_counter
from mip             import OptimizationStatus

In [2]:
from lib.paths     import find_project_root, instance_sort_key

from lib.instances import (
    list_orlibrary_instances     ,
    apply_instance_selection     ,
    load_best_known_costs_to_dict,
)

from lib.metrics   import gap_to_reference_percent
from lib.graph     import read_orlibrary_graph    , all_pairs_shortest_paths
from lib.mip       import extract_open_facilities , compute_solution_cost   , build_pmedian_model
from lib.utils     import finite_or_none          , normalize_number        , parse_optional_int_env

Find the project root directory:

In [3]:
PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

PROJECT_ROOT = /home/rei-luisinho/xopt


### EXPERIMENT CONFIGURATION

In [4]:
INSTANCES_DIR = PROJECT_ROOT  / 'instances'
PMEDOPT_PATH  = INSTANCES_DIR / 'pmedopt.txt'
OUTPUT_DIR    = PROJECT_ROOT  / 'notebooks' / 'experiments_sbpo' / 'artifacts'

EXACT_MIP_OUTPUT_PATH = OUTPUT_DIR / 'mip_exact_results.csv'


TIME_LIMIT_SECONDS = parse_optional_int_env('PMED_TIME_LIMIT_SECONDS') or 180
MAX_INSTANCES      = parse_optional_int_env('PMED_MAX_INSTANCES'     )

INSTANCE_FILTER = os.getenv('PMED_INSTANCE_FILTER')

CANONICAL_INSTANCE_NAMES = [
    f'pmed{i}.txt' for i in range(1, 41)
]

ALL_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES     = apply_instance_selection(
    ALL_INSTANCE_NAMES,
    pattern=INSTANCE_FILTER,
    limit  =MAX_INSTANCES  ,
)
BEST_KNOWN_COSTS = load_best_known_costs_to_dict(PMEDOPT_PATH)

if not INSTANCE_NAMES:
    raise FileNotFoundError(
        f'No p-median instances were selected from {INSTANCES_DIR}.'
    )

In [5]:
print(f'Instances directory  : {INSTANCES_DIR}')
print(f'Instances discovered : {len(ALL_INSTANCE_NAMES)}')
print(f'Instances selected   : {len(INSTANCE_NAMES    )}')

Instances directory  : /home/rei-luisinho/xopt/instances
Instances discovered : 40
Instances selected   : 40


In [6]:
print(f'Time limit per instance (s) : {TIME_LIMIT_SECONDS}')

Time limit per instance (s) : 180


In [7]:
missing_canonical = sorted(set(CANONICAL_INSTANCE_NAMES) - set(ALL_INSTANCE_NAMES      ), key=instance_sort_key)
extra_instances   = sorted(set(ALL_INSTANCE_NAMES      ) - set(CANONICAL_INSTANCE_NAMES), key=instance_sort_key)
missing_optima    = sorted(set(ALL_INSTANCE_NAMES      ) - set(BEST_KNOWN_COSTS        ), key=instance_sort_key)

if missing_canonical:
    print(f'Missing canonical OR-Library instances: {missing_canonical}')
else:
    print('All 40 canonical OR-Library instances are present.')

if extra_instances:
    print(f'Additional instances discovered    : {extra_instances}')

if missing_optima:
    print(f'Instances without best-known value : {missing_optima}')

print()
print('Instances to solve:')
print(', '.join(name.removesuffix('.txt') for name in INSTANCE_NAMES))

All 40 canonical OR-Library instances are present.

Instances to solve:
pmed1, pmed2, pmed3, pmed4, pmed5, pmed6, pmed7, pmed8, pmed9, pmed10, pmed11, pmed12, pmed13, pmed14, pmed15, pmed16, pmed17, pmed18, pmed19, pmed20, pmed21, pmed22, pmed23, pmed24, pmed25, pmed26, pmed27, pmed28, pmed29, pmed30, pmed31, pmed32, pmed33, pmed34, pmed35, pmed36, pmed37, pmed38, pmed39, pmed40


### EXACT MIP

In [8]:
def solve_instance_with_exact_mip(
    instance_name      : str,
    time_limit_seconds : int = TIME_LIMIT_SECONDS,
) -> dict[str, object]:
    instance_path = INSTANCES_DIR / instance_name

    graph            = read_orlibrary_graph(instance_path)
    best_known_value = finite_or_none      (BEST_KNOWN_COSTS.get(instance_name))

    preprocess_start   = perf_counter()
    distances          = all_pairs_shortest_paths(graph['adjacency'])
    preprocess_seconds = perf_counter() - preprocess_start

    build_start   = perf_counter()
    model, x, y   = build_pmedian_model(distances, graph['p'])
    build_seconds = perf_counter() - build_start

    solve_start   = perf_counter()
    status        = model.optimize(max_seconds=time_limit_seconds)
    solve_seconds = perf_counter() - solve_start

    has_incumbent = status in {
        OptimizationStatus.OPTIMAL ,
        OptimizationStatus.FEASIBLE,
    }

    solver_objective_value = finite_or_none(
        model.objective_value if has_incumbent else None
    )

    open_facilities_zero_based: list[int] = []
    validated_objective_value             = None

    if has_incumbent:
        open_facilities_zero_based = extract_open_facilities(y)

        if len(open_facilities_zero_based) != graph['p']:
            raise ValueError(
                f'Expected {graph["p"]} open facilities, but recovered '
                f'{len(open_facilities_zero_based)}.'
            )

        validated_objective_value = compute_solution_cost(
            distances, open_facilities_zero_based
        )

        if solver_objective_value    is not None and \
           validated_objective_value is not None and \
           abs(solver_objective_value - validated_objective_value) > 1e-4:
            raise ValueError(
                 'Solver objective and validated objective do not match: '
                f'{solver_objective_value} vs {validated_objective_value}.'
            )

        if validated_objective_value is not None and \
           best_known_value          is not None and \
           validated_objective_value + 1e-6 < best_known_value:
            raise ValueError(
                 'Validated objective is below the published best-known '
                f'value: {validated_objective_value} < {best_known_value}.'
            )

    objective_value = (
        validated_objective_value
        if   validated_objective_value is not None
        else solver_objective_value
    )

    open_facilities = [
        facility + 1
        for facility in open_facilities_zero_based
    ]

    result = {
        'instance'         : instance_name,
        'n'                : graph['n'],
        'm'                : graph['m'],
        'p'                : graph['p'],
        'best_known_value' : normalize_number(best_known_value),

        'objective_value'           : normalize_number(objective_value),
        'gap_to_best_known_percent' : normalize_number(
            gap_to_reference_percent(objective_value, best_known_value)
        ),

        'matches_best_known': (
            objective_value  is not None and
            best_known_value is not None and
            abs(objective_value - best_known_value) < 1e-6
        ),

        'preprocess_seconds'  : normalize_number(preprocess_seconds           ),
        'model_build_seconds' : normalize_number(build_seconds                ),
        'solve_seconds'       : normalize_number(solve_seconds                ),
        'mip_stage_seconds'   : normalize_number(build_seconds + solve_seconds),
        'total_seconds'       : normalize_number(
            preprocess_seconds + build_seconds + solve_seconds
        ),

        'open_facilities_count' : len     (         open_facilities ),
        'open_facilities'       : ' '.join(map(str, open_facilities)),

        'status' : getattr(status, 'name', str(status)),
        'error'  : '',
    }

    del model
    del x
    del y
    del distances

    gc.collect()

    return result


def attempt_solve_instance_with_exact_mip(
    instance_name      : str,
    time_limit_seconds : int = TIME_LIMIT_SECONDS,
) -> dict[str, object]:
    try:
        return solve_instance_with_exact_mip(
            instance_name,
            time_limit_seconds=time_limit_seconds,
        )
    except Exception as exc:
        gc.collect()

        return {
            'instance'         : instance_name,
            'n'                : None         ,
            'm'                : None         ,
            'p'                : None         ,
            'best_known_value' : normalize_number(
                BEST_KNOWN_COSTS.get(instance_name)
            ),

            'objective_value'           : None ,
            'gap_to_best_known_percent' : None ,
            'matches_best_known'        : False,

            'preprocess_seconds'  : None,
            'model_build_seconds' : None,
            'solve_seconds'       : None,
            'mip_stage_seconds'   : None,
            'total_seconds'       : None,

            'open_facilities_count' : 0 ,
            'open_facilities'       : '',

            'status' : 'ERROR' ,
            'error'  : str(exc),
        }

### APPLY

In [9]:
RESULTS = []

for index, instance_name in enumerate(INSTANCE_NAMES, start=1):
    print(f'[{index:02d}/{len(INSTANCE_NAMES):02d}] Solving {instance_name}...')

    result = attempt_solve_instance_with_exact_mip(
        instance_name,
        time_limit_seconds=TIME_LIMIT_SECONDS,
    )

    RESULTS.append(result)

    print(
        f"  mip_status={  result['status'                   ]},"
        f" mip_obj={      result['objective_value'          ]},"
        f" mip_gap={      result['gap_to_best_known_percent']},"
        f" total_seconds={result['total_seconds'            ]}"
    )

    if result['error']:
        print(f"  error={result['error']}")

RESULTS_DF = pd.DataFrame(RESULTS)

[01/40] Solving pmed1.txt...
  mip_status=OPTIMAL, mip_obj=5819, mip_gap=0, total_seconds=1.3159
[02/40] Solving pmed2.txt...
  mip_status=OPTIMAL, mip_obj=4093, mip_gap=0, total_seconds=1.9544
[03/40] Solving pmed3.txt...
  mip_status=OPTIMAL, mip_obj=4250, mip_gap=0, total_seconds=1.9667
[04/40] Solving pmed4.txt...
  mip_status=OPTIMAL, mip_obj=3034, mip_gap=0, total_seconds=1.1429
[05/40] Solving pmed5.txt...
  mip_status=OPTIMAL, mip_obj=1355, mip_gap=0, total_seconds=1.1798
[06/40] Solving pmed6.txt...
  mip_status=OPTIMAL, mip_obj=7824, mip_gap=0, total_seconds=157.9406
[07/40] Solving pmed7.txt...
  mip_status=OPTIMAL, mip_obj=5631, mip_gap=0, total_seconds=9.2859
[08/40] Solving pmed8.txt...
  mip_status=OPTIMAL, mip_obj=4445, mip_gap=0, total_seconds=9.5144
[09/40] Solving pmed9.txt...
  mip_status=OPTIMAL, mip_obj=2734, mip_gap=0, total_seconds=9.1776
[10/40] Solving pmed10.txt...
  mip_status=OPTIMAL, mip_obj=1255, mip_gap=0, total_seconds=9.4076
[11/40] Solving pmed11.txt.

Saving the results:

In [10]:
OUTPUT_DIR.mkdir (parents=True         , exist_ok=True )
RESULTS_DF.to_csv(EXACT_MIP_OUTPUT_PATH, index   =False)

print(f'Exact MIP results saved to: {EXACT_MIP_OUTPUT_PATH}')

Exact MIP results saved to: /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/mip_exact_results.csv


Displaying the results:

In [11]:
def print_table_summary(
    dataframe     : pd.DataFrame,
    label         : str,
    objective_col : str,
    gap_col       : str,
    time_col      : str,
) -> None:
    solved_count  = int(dataframe[objective_col].notna().sum())
    matches_count = int(
        (dataframe[gap_col].fillna(np.inf).abs() < 1e-6).sum()
    )

    mean_time = dataframe[time_col].dropna().mean()
    mean_gap  = dataframe[gap_col ].dropna().mean()

    print(label)
    print(f'  solved instances            : {solved_count }/{len(dataframe)}')
    print(f'  matches best-known          : {matches_count}/{len(dataframe)}')
    print(f'  mean time (s)               : {normalize_number(mean_time)}')
    print(f'  mean gap to best-known (%)  : {normalize_number(mean_gap )}')
    print()


print_table_summary(
    RESULTS_DF,
    label        ='Exact MIP summary:',
    objective_col='objective_value'   ,
    gap_col ='gap_to_best_known_percent',
    time_col='total_seconds'            ,
)

Exact MIP summary:
  solved instances            : 30/40
  matches best-known          : 29/40
  mean time (s)               : 94.2887
  mean gap to best-known (%)  : 0.0048



Approach overview:

In [12]:
EXACT_MIP_DISPLAY_DF = RESULTS_DF[
    [
        'instance'                 ,
        'best_known_value'         ,
        'objective_value'          ,
        'gap_to_best_known_percent',

        'solve_seconds',
        'total_seconds',
        'status'       ,
    ]
].copy()

In [13]:
APPROACH_SUMMARY_DF = pd.DataFrame(
    [
        {
            'approach'  : 'Exact MIP'    ,
            'instances' : len(RESULTS_DF),

            'mean_runtime_seconds'           : normalize_number(RESULTS_DF['total_seconds'            ].dropna().mean()),
            'mean_gap_to_best_known_percent' : normalize_number(RESULTS_DF['gap_to_best_known_percent'].dropna().mean()),
        }
    ]
)

display(APPROACH_SUMMARY_DF)

,approach,instances,mean_runtime_seconds,mean_gap_to_best_known_percent
0,Exact MIP,40,94.2887,0.0048


Exact MIP per-instance results:

In [14]:
display(EXACT_MIP_DISPLAY_DF)

,instance,best_known_value,objective_value,gap_to_best_known_percent,solve_seconds,total_seconds,status
0,pmed1.txt,5819,5819.0,0.0000,1.1433,1.3159,OPTIMAL
1,pmed2.txt,4093,4093.0,0.0000,1.8536,1.9544,OPTIMAL
2,pmed3.txt,4250,4250.0,0.0000,1.8650,1.9667,OPTIMAL
3,pmed4.txt,3034,3034.0,0.0000,1.0306,1.1429,OPTIMAL
4,pmed5.txt,1355,1355.0,0.0000,1.0880,1.1798,OPTIMAL
5,pmed6.txt,7824,7824.0,0.0000,157.4519,157.9406,OPTIMAL
6,pmed7.txt,5631,5631.0,0.0000,8.9024,9.2859,OPTIMAL
7,pmed8.txt,4445,4445.0,0.0000,9.1346,9.5144,OPTIMAL
8,pmed9.txt,2734,2734.0,0.0000,8.8001,9.1776,OPTIMAL
9,pmed10.txt,1255,1255.0,0.0000,9.0433,9.4076,OPTIMAL
